# Modeling — Random Forest (quantile 0.9)

Desain: `docs/superpowers/specs/2026-08-18-random-forest-modeling-design.md`.
Rencana: `docs/superpowers/plans/2026-08-18-random-forest-modeling.md`.

Notebook ini tipis dengan sengaja. Semua logika ada di `utils/walk_forward.py`
dan `utils/model_random_forest.py`, supaya jalur skrip dan jalur notebook tidak
bisa berbeda — persis kesalahan yang pernah terjadi di `data-processing.ipynb`.

**Desember 2025 terkunci** dan tidak dinilai di sini.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from utils import evaluation, model_random_forest as rf
from utils import modeling_prep, walk_forward

df = pd.read_parquet(modeling_prep.MODEL_INPUT_FILE)
print(f"{len(df):,} rows x {df.shape[1]} columns")

## Benchmark

Satu fit di training set penuh fold 5, untuk memastikan batas leaf storage
berlaku dan menentukan ukuran pencarian. Angkanya dicatat di
`docs/hasil-modeling-rf.md`.

In [ ]:
import resource
import time

split = walk_forward.prepare_fold(df, 5)
train, valid = split["train"], split["valid"]
print(f"train {len(train):,} rows, valid {len(valid):,} rows")

params = dict(rf.DEFAULT_PARAMS)
print("estimated leaf storage: "
      f"{rf.estimate_leaf_memory_bytes(params, len(train)) / 1024 ** 3:.2f} GB")

start = time.time()
prediction = rf.make_fit_predict(params)(train, valid)
elapsed = time.time() - start

peak_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss  # bytes on macOS
print(f"wall time {elapsed / 60:.1f} min")
print(f"peak RSS  {peak_bytes / 1024 ** 3:.2f} GB")
print(f"prediction mean {prediction.mean():.2f}, max {prediction.max():.2f}")

## Pencarian hyperparameter

18 kandidat, tersaring budget memori, dinilai di fold 3 dan 5 dengan pinball@0.9
gabungan. Hasil yang dilaporkan datang dari walk-forward lima fold di bawah,
bukan dari sini — menilai di fold yang memilih pemenang akan optimistis.

In [ ]:
train_size = len(walk_forward.prepare_fold(df, 5)["train"])
candidates = rf.sample_search_space(18, n_train=train_size, seed=42)
search_results = rf.run_search(df, candidates, folds=rf.SEARCH_FOLDS,
                               checkpoint_path=rf.SEARCH_FILE)
search_results.to_csv(rf.SEARCH_FILE, index=False)
search_results.sort_values("pinball").head(10)

## Walk-forward final

Konfigurasi pemenang di kelima fold, melawan ketiga baseline naive pada baris
yang identik.

In [ ]:
best = rf.select_best(search_results, candidates)
rf.save_best_params(best)
print(best)

fit_predict = rf.make_fit_predict(best)
results = walk_forward.run_walk_forward(df, fit_predict, model_name="random_forest")
results.to_csv(rf.RESULTS_FILE, index=False)

overall = results[results["group_col"].isna()]
overall.pivot_table(index="model", columns="fold_id", values="pinball").round(3)

In [ ]:
bundle = rf.fit_final(df, best)
rf.save_bundle(bundle)
print(f"trained on {bundle['n_train']:,} rows, "
      f"{len(bundle['columns'])} columns, quantile {bundle['quantile']}")

## Hasil

Tiga potongan, masing-masing melawan ketiga baseline naive pada baris identik.
Satu angka global menyesatkan di data yang 45% targetnya nol.

In [ ]:
results = pd.read_csv(rf.RESULTS_FILE)

print("=== per fold (overall) ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

print("\n=== coverage and fill rate (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage'):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate'):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units'):9.1f}")